# 01 — Building Euclidean Tilings

Eucare builds tilings by growing rings of *prototiles* outward from a seed face. This notebook covers:

- the three regular Platonic tilings,
- the Archimedean tilings via convenient factories,
- a quick taste of Conway operators (dual, ambo, kis, truncate, gyro).

In [ ]:
import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    plotting,
    reciprocal_figures,
    rendering,
)


def plot_g(G, ax=None, color='black', linewidth=1.0):
    """Draw the edges of a half-edge graph G on `ax` (or the current axes)."""
    if ax is None:
        ax = plt.gca()
    lines = np.array([
        [G.geometry.to_euclidean(h.orig['pos']),
         G.geometry.to_euclidean(h.dest['pos'])]
        for h in G.halfedges_representing_edges()
    ])
    plotting.plot_lines(lines, ax=ax, colors=color, linewidths=linewidth)
    plotting.set_equal_aspect(ax)
    ax.axis('off')


## The three regular tilings

`platonic(n)` returns a list of prototiles that tile the plane regularly for `n ∈ {3, 4, 6}` (triangles, squares, hexagons).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, n in zip(axes, [3, 4, 6]):
    G = example_graphs.from_tiles(example_tilesets.platonic(n), rings=3)
    G.recompute_lengths_and_angles()
    plot_g(G, ax=ax)
    ax.set_title(f'platonic({n})')
plt.tight_layout(); plt.show()


## Archimedean tilings

`example_tilesets` provides several Archimedean tilings as named factories. Each returns a list of prototiles you pass to `from_tiles`.

In [ ]:
factories = [
    ('3.12.12',     example_tilesets.t_3_12_12),
    ('4.6.12',      example_tilesets.t_4_6_12),
    ('3.3.4.3.4',   example_tilesets.t_3_3_4_3_4),
    ('3.3.3.3.6',   example_tilesets.t_3_3_3_3_6),
]
fig, axes = plt.subplots(1, len(factories), figsize=(4 * len(factories), 4))
for ax, (name, factory) in zip(axes, factories):
    G = example_graphs.from_tiles(factory(), rings=2)
    G.recompute_lengths_and_angles()
    plot_g(G, ax=ax)
    ax.set_title(name)
plt.tight_layout(); plt.show()


## Conway operators

Conway operators take a tiling and return a new one by replacing each face with a small canonical pattern. They're a quick way to generate interesting CP candidates for SRG.

Common ones in `eucare.conway`:

| Operator | Effect |
|----------|--------|
| `dual_graph()`     | Dual: vertex per face, face per vertex |
| `ambo_graph()`     | Cuts off corners at edge midpoints |
| `kis_graph()`      | Adds a centre vertex to each face |
| `truncate_graph()` | Truncates corners |
| `gyro_graph()`     | Gyro: chiral subdivision |
| `starify_graph()`  | Stars each face |


In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(6), rings=3)
G.recompute_lengths_and_angles()

ops = [
    ('original', None),
    ('dual',     conway.dual_graph()),
    ('ambo',     conway.ambo_graph()),
    ('kis',      conway.kis_graph()),
    ('truncate', conway.truncate_graph()),
    ('gyro',     conway.gyro_graph()),
]
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, (name, op) in zip(axes.ravel(), ops):
    H = G.copy() if op is None else op(G.copy(), delete_on_border=True)
    H.recompute_lengths_and_angles()
    plot_g(H, ax=ax)
    ax.set_title(name)
plt.tight_layout(); plt.show()


## What's next

See [`02_Curved_Geometries`](02_Curved_Geometries.ipynb) for spherical and hyperbolic versions of the same construction. The interactive [`Tiling Demo`](Tiling%20Demo.ipynb) notebook is a deeper companion.